<a href="https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [ ]:
 print(len(manual_token), manual_token[:3])

145 HF_


In [ ]:
print(len(manual_token))
print(repr(manual_token[:6]))
print('CREATE' in manual_token, 'getpass' in manual_token, chr(10) in manual_token)

145
'HF_TOK'
True False False


In [ ]:
# in the getpass cell, paste this token when prompted
manual_token = getpass.getpass("Paste your HF token here: ")

Paste your HF token here: ··········


In [ ]:
print(len(manual_token), manual_token[:3])

37 hf_


In [ ]:
HF_TOKEN = manual_token
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}');")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)   # use whatever variable name cell 11 actually assigned
print(api.whoami()["name"])
print(api.dataset_info("FlyRank/internship-warehouse"))

hashss55
DatasetInfo(id='FlyRank/internship-warehouse', author='FlyRank', card_data={'annotations_creators': None, 'language_creators': None, 'language': ['en'], 'license': 'other', 'multilinguality': None, 'size_categories': ['10M<n<100M'], 'source_datasets': None, 'task_categories': None, 'task_ids': None, 'paperswithcode_id': None, 'pretty_name': 'FlyRank Internship — Warehouse Star Schema (Pseudonymized, Gated)', 'config_names': None, 'train_eval_index': None, 'tags': ['seo', 'content-performance', 'data-warehouse', 'tabular', 'education', 'flyrank-internship'], 'extra_gated_prompt': 'By requesting access you agree to the FlyRank Internship Data Use Terms: anonymized research and education use only; no attempt to re-identify clients, domains, queries, keywords, or content; no redistribution of the raw data; and no client-identifying data in any public output (case study, repo, chart, or demo).', 'extra_gated_fields': {'Name': 'text', 'Email': 'text', 'Affiliation or cohort': 'text'

In [ ]:
query = f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}');"

In [ ]:
print(repr(query))

'CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN \'HF_TOKEN = manual_token  # point the old variable at the working token con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN \'{HF_TOKEN}\');")\');'


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [ ]:
print(TABLES)

{'dim_clients': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')", 'dim_content': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')", 'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')", 'fact_daily_sample': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')", 'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


In [ ]:
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df())
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily_sample']}").df())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [ ]:
q = f"""
WITH recent AS (
    SELECT *
    FROM {TABLES['fact_daily_sample']}
    WHERE gsc_data_available = true
),
agg AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks)::DOUBLE / SUM(gsc_impressions) END AS ctr,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_sum_position)::DOUBLE / SUM(gsc_impressions) END AS avg_position,
        SUM(ga4_pageviews) AS pageviews,
        SUM(ga4_sessions) AS sessions,
        SUM(ga4_engaged_sessions) AS engaged_sessions,
        SUM(scroll_events) AS scroll_events,
        COUNT(DISTINCT report_date) AS days_with_data
    FROM recent
    GROUP BY content_hash_id, client_hash_id
)
SELECT a.*, c.word_count, c.content_type, c.main_intent,
       c.is_published, c.is_deleted, c.backlinks, c.search_volume
FROM agg a
JOIN {TABLES['dim_content']} c USING (content_hash_id)
WHERE c.is_published = true AND c.is_deleted = false
"""
feat = con.sql(q).df()
print(feat.shape)
feat.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(198480, 18)


,content_hash_id,client_hash_id,impressions,clicks,ctr,avg_position,pageviews,sessions,engaged_sessions,scroll_events,days_with_data,word_count,content_type,main_intent,is_published,is_deleted,backlinks,search_volume
0,content_3f89163edebe3dcd,client_3ffa76342f366962,1.0,0.0,0.0,7.000000,0.0,0.0,0.0,0.0,1,<NA>,keyword article,informational,True,False,<NA>,0
1,content_751cf2041321dc55,client_3ffa76342f366962,6.0,0.0,0.0,19.000000,0.0,0.0,0.0,0.0,6,<NA>,keyword article,informational,True,False,<NA>,0
2,content_f4e77335eb90e5c6,client_3ffa76342f366962,10.0,0.0,0.0,47.100000,0.0,0.0,0.0,0.0,9,<NA>,keyword article,None,True,False,<NA>,<NA>
3,content_b09dcb6310544681,client_3ffa76342f366962,1.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1,<NA>,keyword article,informational,True,False,<NA>,0
4,content_1bae02f58b51e748,client_3ffa76342f366962,4.0,0.0,0.0,7.000000,0.0,0.0,0.0,0.0,3,<NA>,keyword article,informational,True,False,<NA>,0
5,content_c86e6a67153ff7b1,client_3ffa76342f366962,2.0,0.0,0.0,8.000000,0.0,0.0,0.0,0.0,1,<NA>,keyword article,informational,True,False,<NA>,0
6,content_f573c53b853ba00b,client_3ffa76342f366962,3.0,0.0,0.0,4.000000,0.0,0.0,0.0,0.0,2,<NA>,keyword article,informational,True,False,<NA>,0
7,content_117ab9a954e3414c,client_3ffa76342f366962,54.0,0.0,0.0,4.240741,0.0,0.0,0.0,0.0,20,<NA>,keyword article,informational,True,False,<NA>,0
8,content_d43df6ee98b55221,client_3ffa76342f366962,1.0,0.0,0.0,3.000000,0.0,0.0,0.0,0.0,1,<NA>,keyword article,informational,True,False,<NA>,0
9,content_ad5ea1bf5181e833,client_3ffa76342f366962,2.0,0.0,0.0,4.500000,0.0,0.0,0.0,0.0,2,<NA>,keyword article,informational,True,False,<NA>,0


In [ ]:
print("rows:", len(feat))
print("\nimpressions describe:\n", feat['impressions'].describe())
print("\nctr describe:\n", feat['ctr'].describe())
print("\npct impressions == 0:", (feat['impressions'] == 0).mean())
print("pct clicks == 0:", (feat['clicks'] == 0).mean())
print("pct avg_position == 0:", (feat['avg_position'] == 0).mean())
print("\nword_count null rate:", feat['word_count'].isna().mean())
print("search_volume null rate:", feat['search_volume'].isna().mean())
print("main_intent null rate:", feat['main_intent'].isna().mean())

rows: 198480

impressions describe:
 count    198480.000000
mean       1073.733459
std        5155.144607
min           1.000000
25%          13.000000
50%         108.000000
75%         553.000000
max      615012.000000
Name: impressions, dtype: float64

ctr describe:
 count    198480.000000
mean          0.004479
std           0.029740
min           0.000000
25%           0.000000
50%           0.000000
75%           0.003583
max           1.000000
Name: ctr, dtype: float64

pct impressions == 0: 0.0
pct clicks == 0: 0.607884925433293
pct avg_position == 0: 0.015114873035066506

word_count null rate: 0.24619609028617492
search_volume null rate: 0.046508464328899636
main_intent null rate: 0.05125957275292221


In [ ]:
import pandas as pd

work = feat[feat['avg_position'] > 0].copy()

bins = [0, 3, 5, 10, 20, 50, 100, float('inf')]
labels = ['1-3', '4-5', '6-10', '11-20', '21-50', '51-100', '100+']
work['position_bucket'] = pd.cut(work['avg_position'], bins=bins, labels=labels)

curve = work.groupby('position_bucket', observed=True).agg(
    pages=('content_hash_id', 'count'),
    impressions=('impressions', 'sum'),
    clicks=('clicks', 'sum'),
)
curve['expected_ctr'] = curve['clicks'] / curve['impressions']
print(curve)

                 pages  impressions    clicks  expected_ctr
position_bucket                                            
1-3               7245    8376926.0   48959.0      0.005845
4-5              15639   35493954.0  191668.0      0.005400
6-10             62082  119882986.0  411724.0      0.003434
11-20            41014   27227539.0  109212.0      0.004011
21-50            44016   19052131.0   50263.0      0.002638
51-100           25162    3036384.0    2101.0      0.000692
100+               322      40828.0      97.0      0.002376


In [ ]:
bins2 = [0, 3, 5, 10, 20, 50, float('inf')]
labels2 = ['1-3', '4-5', '6-10', '11-20', '21-50', '51+']
work['position_bucket'] = pd.cut(work['avg_position'], bins=bins2, labels=labels2)

curve2 = work.groupby('position_bucket', observed=True).agg(
    pages=('content_hash_id', 'count'),
    impressions=('impressions', 'sum'),
    clicks=('clicks', 'sum'),
)
curve2['expected_ctr'] = curve2['clicks'] / curve2['impressions']
print(curve2)

                 pages  impressions    clicks  expected_ctr
position_bucket                                            
1-3               7245    8376926.0   48959.0      0.005845
4-5              15639   35493954.0  191668.0      0.005400
6-10             62082  119882986.0  411724.0      0.003434
11-20            41014   27227539.0  109212.0      0.004011
21-50            44016   19052131.0   50263.0      0.002638
51+              25484    3077212.0    2198.0      0.000714


In [ ]:
benchmark_ctr = curve2['expected_ctr'].to_dict()
print(benchmark_ctr)

{'1-3': 0.005844506684194178, '4-5': 0.005400018267899936, '6-10': 0.003434382256711557, '11-20': 0.00401108598173342, '21-50': 0.0026381825739073494, '51+': 0.0007142829288329825}


In [ ]:
work['expected_ctr'] = work['position_bucket'].map(benchmark_ctr).astype(float)
work['ctr_gap'] = work['expected_ctr'] - work['ctr']  # positive = underperforming for its position
work['opportunity_score'] = work['ctr_gap'] * work['impressions']  # weight the gap by traffic volume

# reason codes
def reason_code(row):
    if row['ctr_gap'] <= 0:
        return "meeting_or_beating_expected_ctr"
    if row['impressions'] < 100:
        return "gap_but_low_volume"
    return "underperforming_for_position"

work['reason_code'] = work.apply(reason_code, axis=1)

ranked = work.sort_values('opportunity_score', ascending=False)
print(ranked[['content_hash_id', 'client_hash_id', 'position_bucket', 'ctr', 'expected_ctr',
              'ctr_gap', 'impressions', 'opportunity_score', 'reason_code']].head(20))

                 content_hash_id           client_hash_id position_bucket  \
40342   content_943dc881428182b8  client_8ddc46da5414ffd8             1-3   
128037  content_acbcc847f8996314  client_62f4a7e64f5e0096             4-5   
40603   content_32c5cc913fb4ff41  client_8ddc46da5414ffd8             4-5   
139158  content_d0acf7062bc6b257  client_8ddc46da5414ffd8             1-3   
106615  content_9ef3d7516483e665  client_e547b89c05043229             1-3   
49948   content_c1f764a2f362d1c3  client_a80fca3f171ed1de            6-10   
41156   content_c15dabdcf2be1866  client_8ddc46da5414ffd8             4-5   
106455  content_21309e9a83c83653  client_e547b89c05043229             4-5   
128011  content_f352b7cfd0b2f434  client_62f4a7e64f5e0096             4-5   
172555  content_012de75c008aa653  client_a80fca3f171ed1de            6-10   
31345   content_33d31496fca9665e  client_73cda7b4e4f265ea            6-10   
148593  content_9540d884af3e41fd  client_a80fca3f171ed1de            6-10   

That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.555     0.345     0.425      9389
           1      0.688     0.840     0.756     16162

    accuracy                          0.658     25551
   macro avg      0.622     0.592     0.591     25551
weighted avg      0.639     0.658     0.635     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Group by client this time, instead of random row split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_data, groups=model_data['client_hash_id']))

X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(model_data['client_hash_id'].iloc[train_idx])
test_clients = set(model_data['client_hash_id'].iloc[test_idx])
print("Client overlap between train/test (should be 0):", len(train_clients & test_clients))

model_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_g, y_tr_g)

print(f"\nGrouped-by-client split:")
print(f"base rate (always predict majority): {max(y_te_g.mean(), 1 - y_te_g.mean()):.3f}")
print(classification_report(y_te_g, model_grouped.predict(X_te_g), digits=3))

Client overlap between train/test (should be 0): 0

Grouped-by-client split:
base rate (always predict majority): 0.677
              precision    recall  f1-score   support

           0      0.413     0.466     0.438     16706
           1      0.729     0.684     0.706     34996

    accuracy                          0.614     51702
   macro avg      0.571     0.575     0.572     51702
weighted avg      0.627     0.614     0.619     51702



## Honest finding: naive split overstated the model

The first pass (random row split) showed the model beating the base rate — 0.658 vs. 0.633
accuracy, a real-looking ~2.5-point lift. Under a client-grouped split (no client appears in
both train and test), that lift disappeared entirely: the model (0.614 accuracy) actually
underperformed the base rate (0.677).

This means the original result was very likely driven by client-level leakage, not genuine
content-level signal. Random splitting let the same clients appear in both train and test,
so the model could partially learn "this looks like client X's typical behavior" rather than
learning a pattern that transfers to unseen clients or content.

Conclusion: with the five features tested (imp_prev30, visible_queries, rare_share, anon_share,
top_query_share), there is no evidence of a real, generalizable signal for predicting a >20%
month-over-month impression decline. This is a directional, decision-support finding, not a
final verdict — a different feature set (position volatility, trend history beyond one prior
window, or content-level attributes like word_count/content_type) might carry more signal,
but that's future work, not something these features support as-is.

This is a stronger, more honest result to report than the misleadingly good 0.658 number would
have been — it demonstrates the value of the grouped validation check, not just a model that
"works."